In [ ]:
#!/usr/bin/env python3

"""
run_5nn_classification_table.py

Driver script for reproducing the 5-NN classification table
for the six DNA representation methods.

The script:
1. Loads each NCBI dataset metadata file.
2. Loads the precomputed distance matrix for each method.
3. Filters families with at least 15 samples.
4. Performs 5-fold stratified cross-validation.
5. Repeats the experiment over 30 random seeds.
6. Computes ACC, BA, macro-F1, macro-Recall, and macro-Precision.
7. Displays a clean pandas table in Jupyter.
8. Saves the numerical results to CSV.

Methods:
    CAKR, NVM, FFP-JS, FFP-KL, MKS, FPS


"""

import os
import random
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


# ============================================================
# Seed setting
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)


# ============================================================
# Label filtering
# ============================================================

def find_labels_with_min_count(labels, min_count=15):
    """
    Return indices of samples whose labels occur at least min_count times.
    """

    label_counts = Counter(labels)

    valid_labels = {
        label for label, count in label_counts.items()
        if count >= min_count
    }

    indices = [
        i for i, label in enumerate(labels)
        if label in valid_labels
    ]

    return indices


# ============================================================
# Classification metrics
# ============================================================

def compute_classification_metrics(y_true, y_pred, average="macro"):
    """
    Compute classification metrics.
    """

    return {
        "ACC": accuracy_score(y_true, y_pred),
        "BA": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, average=average, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, average=average, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, average=average, zero_division=0
        ),
    }


# ============================================================
# 5-fold 5-NN from precomputed distance matrix
# ============================================================

def five_fold_5nn(distance_matrix, labels, n_neighbors=5, random_state=42):
    """
    Perform 5-fold stratified cross-validation using a precomputed
    distance matrix and 5-NN classification.
    """

    distance_matrix = np.asarray(distance_matrix)
    labels = np.asarray(labels)

    N = distance_matrix.shape[0]

    assert distance_matrix.shape == (N, N), "distance_matrix must be NxN"
    assert len(labels) == N, "labels length must match distance_matrix size"

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state,
    )

    fold_metrics_list = []

    for train_idx, test_idx in skf.split(X=labels, y=labels):
        fold_preds = []
        fold_true = []

        for test_sample in test_idx:
            dist_row = distance_matrix[test_sample].copy()

            # Exclude the test sample itself
            dist_row[test_sample] = np.inf

            # Only consider training samples
            train_distances = dist_row[train_idx]

            # Nearest neighbors inside training set
            nn_in_train = np.argpartition(train_distances, n_neighbors)[:n_neighbors]
            nn_global_idx = train_idx[nn_in_train]

            neighbor_labels = labels[nn_global_idx]

            # Majority vote
            chosen_label = Counter(neighbor_labels).most_common(1)[0][0]

            fold_preds.append(chosen_label)
            fold_true.append(labels[test_sample])

        fold_metrics = compute_classification_metrics(
            fold_true,
            fold_preds,
            average="macro",
        )

        fold_metrics_list.append(fold_metrics)

    # Average over folds
    avg_metrics = {}

    for key in fold_metrics_list[0].keys():
        avg_metrics[key] = np.mean([m[key] for m in fold_metrics_list])

    return avg_metrics


# ============================================================
# Repeat over multiple seeds
# ============================================================

def multiple_seeds_5nn(distance_matrix, labels, seeds, n_neighbors=5):
    """
    Run 5-fold 5-NN for multiple random seeds and average the metrics.
    """

    all_seed_metrics = []

    for seed in seeds:
        seed_metrics = five_fold_5nn(
            distance_matrix=distance_matrix,
            labels=labels,
            n_neighbors=n_neighbors,
            random_state=seed,
        )

        all_seed_metrics.append(seed_metrics)

    avg_overall = {}

    for key in all_seed_metrics[0].keys():
        avg_overall[key] = np.mean([m[key] for m in all_seed_metrics])

    return avg_overall


# ============================================================
# Distance-file paths
# ============================================================

def get_distance_path(method_key, data_name, k):
    """
    Return the distance-matrix path for each method.

    Adjust this function only if your repository uses different names.
    """

    if method_key == "CAKR":
        encoder_folder = "CAKR"
        return f"distances/{encoder_folder}/{data_name}/k{k}_distance_facet0.npy"

    if method_key == "NVM":
        return f"distances/NVM/{data_name}/k{k}_distance.npy"

    if method_key == "FFP-JS":
        return f"distances/FFP-JS/{data_name}/k{k}_distance.npy"

    if method_key == "FFP-KL":
        return f"distances/FFP-KL/{data_name}/k{k}_distance.npy"

    if method_key == "MKS":
        return f"distances/MKS/{data_name}/k{k}_distance.npy"

    if method_key == "FPS":
        # FPS usually does not require k.
        # First try k-independent path, then k-dependent path.
        path1 = f"distances/FPS/{data_name}/distance.npy"
        path2 = f"distances/FPS/{data_name}/k{k}_distance.npy"

        if os.path.exists(path1):
            return path1

        return path2

    raise ValueError(f"Unknown method: {method_key}")


# ============================================================
# Load labels
# ============================================================

def load_dataset_labels(data_name, min_count=15):
    """
    Load accessions and labels, sort by accession, filter by min_count,
    and return filtered indices and integer labels.
    """

    csv_path = f"data/{data_name}.csv"

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Missing dataset CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    x_list = df["Accession (version)"].to_list()
    y_list = df["Family"].to_list()

    # Sort to match the order used when distance matrices were created
    x_list, y_list = zip(*sorted(zip(x_list, y_list)))
    x_list = list(x_list)
    y_list = list(y_list)

    indices = find_labels_with_min_count(
        y_list,
        min_count=min_count,
    )

    y_filtered = [y_list[i] for i in indices]

    label_encoder = LabelEncoder()
    labels = label_encoder.fit_transform(y_filtered)
    labels = np.array(labels, dtype=np.int32)

    return indices, labels, x_list, y_list


# ============================================================
# Configuration
# ============================================================

datasets = {
    "NCBI 2020": "Yau2020_record_processed",
    "NCBI 2022": "Yau2022_record_processed",
    "NCBI 2024": "NCBI_record_valid_nucleotide",
    "NCBI 2024 All": "NCBI_record_valid_count",
}

methods = [
    "CAKR",
    "NVM",
    "FFP-JS",
    "FFP-KL",
    "MKS",
    "FPS",
]

# In the manuscript classification task, k=4 was selected.
method_k = {
    "CAKR": 4,
    "NVM": 5,
    "FFP-JS": 3,
    "FFP-KL": 3,
    "MKS": 3,
    "FPS": 3,
}

min_count = 15
n_neighbors = 5
seeds = np.arange(1, 31)

output_csv = "five_nn_classification_results.csv"


# ============================================================
# Main computation
# ============================================================

set_seed(42)

results = []

for dataset_label, data_name in datasets.items():
    print("\n===================================================")
    print(f"Dataset: {dataset_label}")
    print(f"Data file key: {data_name}")
    print("===================================================")

    try:
        indices, labels, x_list, y_list = load_dataset_labels(
            data_name=data_name,
            min_count=min_count,
        )
    except FileNotFoundError as e:
        print(e)
        continue

    print("Original sample size:", len(y_list))
    print("Filtered sample size:", len(labels))
    print("Number of classes:", len(np.unique(labels)))

    for method in methods:
        k = method_k[method]
        distance_path = get_distance_path(method, data_name, k)

        print(f"\nMethod: {method}")
        print(f"k: {k}")
        print("Distance path:", distance_path)

        if not os.path.exists(distance_path):
            print("Missing distance file. Skipping.")

            results.append({
                "Data": dataset_label,
                "Method": method,
                "k": k,
                "ACC": np.nan,
                "BA": np.nan,
                "F1": np.nan,
                "Recall": np.nan,
                "Precision": np.nan,
                "Distance_file": distance_path,
                "Status": "missing_distance_file",
            })

            continue

        distance_matrix = np.load(distance_path)

        if distance_matrix.shape[0] != len(y_list):
            print("Shape mismatch. Skipping.")
            print("Distance matrix shape:", distance_matrix.shape)
            print("Number of labels before filtering:", len(y_list))

            results.append({
                "Data": dataset_label,
                "Method": method,
                "k": k,
                "ACC": np.nan,
                "BA": np.nan,
                "F1": np.nan,
                "Recall": np.nan,
                "Precision": np.nan,
                "Distance_file": distance_path,
                "Status": "shape_mismatch",
            })

            continue

        subD = distance_matrix[np.ix_(indices, indices)]

        metrics_result = multiple_seeds_5nn(
            distance_matrix=subD,
            labels=labels,
            seeds=seeds,
            n_neighbors=n_neighbors,
        )

        row = {
            "Data": dataset_label,
            "Method": method,
            "k": k,
            "ACC": metrics_result["ACC"],
            "BA": metrics_result["BA"],
            "F1": metrics_result["F1"],
            "Recall": metrics_result["Recall"],
            "Precision": metrics_result["Precision"],
            "Distance_file": distance_path,
            "Status": "ok",
        }

        results.append(row)

        print("Average metrics across 30 seeds:")
        print(f"ACC:       {metrics_result['ACC']:.4f}")
        print(f"BA:        {metrics_result['BA']:.4f}")
        print(f"F1:        {metrics_result['F1']:.4f}")
        print(f"Recall:    {metrics_result['Recall']:.4f}")
        print(f"Precision: {metrics_result['Precision']:.4f}")


# ============================================================
# Save raw results
# ============================================================

results_df = pd.DataFrame(results)

results_df.to_csv(output_csv, index=False)

print("\nSaved numerical results to:")
print(output_csv)


# ============================================================
# Make clean Jupyter table
# ============================================================

table = results_df[
    ["Data", "Method", "ACC", "BA", "F1", "Recall", "Precision"]
].copy()

# Keep manuscript row and method order
data_order = list(datasets.keys())
method_order = methods

table["Data"] = pd.Categorical(
    table["Data"],
    categories=data_order,
    ordered=True,
)

table["Method"] = pd.Categorical(
    table["Method"],
    categories=method_order,
    ordered=True,
)

table = table.sort_values(["Data", "Method"])

# Round as in manuscript
table_rounded = table.copy()

for col in ["ACC", "BA", "F1", "Recall", "Precision"]:
    table_rounded[col] = table_rounded[col].round(3)

print("\n===================================================")
print("5-NN classification table")
print("===================================================")

display(table_rounded)